In [ ]:
import pandas as pd
import numpy as np

# Exact schedule from your DB for lab E-312
# (day: 0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri, 5=Sat)
schedule = [
    # day, start_h, start_m, end_h, end_m, subject, batch
    (0,  9, 15, 11, 15, 'AQC',      'TA2'),   # Monday
    (0, 11, 30, 13, 30, 'AQC',      'TB2'),
    (1,  9, 15, 11, 15, 'AQC',      'TB3'),   # Tuesday
    (1, 11, 30, 13, 30, 'PE-II DA', 'D3'),
    (2,  9, 15, 11, 15, 'AQC',      'TB1'),   # Wednesday
    (2, 11, 30, 13, 30, 'PE-II VCC','V1'),
    (3,  9, 15, 11, 15, 'AQC',      'TB4'),   # Thursday
    (3, 14, 15, 16, 15, 'AQC',      'TA3'),
    (4,  9, 15, 11, 15, 'VCC',      'V2'),    # Friday
    (4, 14, 15, 16, 15, 'AQC',      'TA4'),
    (5,  9, 15, 11, 15, 'MP&S[H]',  'BTech'), # Saturday
]

records = []
for (day, sh, sm, eh, em, subj, batch) in schedule:
    start_mins = sh * 60 + sm
    end_mins   = eh * 60 + em
    for t in range(start_mins, end_mins, 5):
        records.append({
            "time_mins": t,
            "day":       day,
            "subject":   subj,
            "batch":     batch
        })

df = pd.DataFrame(records)
print(f"Total training samples: {df.shape[0]}")
print(df['subject'].value_counts())

Total training samples: 264
subject
AQC          168
PE-II DA      24
PE-II VCC     24
VCC           24
MP&S[H]       24
Name: count, dtype: int64


In [2]:
#Train KNN model:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df[["time_mins", "day"]]
y = df["subject"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%")
print()
print(classification_report(y_test, y_pred))

Accuracy: 62.3%

              precision    recall  f1-score   support

         AQC       0.63      1.00      0.78        33
     MP&S[H]       0.00      0.00      0.00         5
    PE-II DA       0.00      0.00      0.00         5
   PE-II VCC       0.00      0.00      0.00         5
         VCC       0.00      0.00      0.00         5

    accuracy                           0.62        53
   macro avg       0.13      0.20      0.16        53
weighted avg       0.40      0.62      0.48        53



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
#Test predictions manually
test_cases = [
    (9*60+30,  0, "Monday 9:30    → expect AQC/TA2"),
    (12*60+0,  0, "Monday 12:00   → expect AQC/TB2"),
    (10*60+0,  1, "Tuesday 10:00  → expect AQC/TB3"),
    (12*60+30, 1, "Tuesday 12:30  → expect PE-II DA"),
    (10*60+0,  2, "Wednesday 10:00→ expect AQC/TB1"),
    (12*60+0,  3, "Thursday 12:00 → expect AQC/TB4"),
    (15*60+0,  3, "Thursday 15:00 → expect AQC/TA3"),
    (10*60+0,  5, "Saturday 10:00 → expect MP&S[H]"),
]

for (t, d, desc) in test_cases:
    sample = pd.DataFrame([[t, d]], columns=["time_mins","day"])
    pred   = model.predict(sample)[0]
    conf   = model.predict_proba(sample).max() * 100
    print(f"{desc}")
    print(f"  → Predicted: {pred}  (confidence: {conf:.0f}%)\n")

Monday 9:30    → expect AQC/TA2
  → Predicted: AQC  (confidence: 100%)

Monday 12:00   → expect AQC/TB2
  → Predicted: AQC  (confidence: 33%)

Tuesday 10:00  → expect AQC/TB3
  → Predicted: AQC  (confidence: 100%)

Tuesday 12:30  → expect PE-II DA
  → Predicted: AQC  (confidence: 33%)

Wednesday 10:00→ expect AQC/TB1
  → Predicted: AQC  (confidence: 100%)

Thursday 12:00 → expect AQC/TB4
  → Predicted: AQC  (confidence: 33%)

Thursday 15:00 → expect AQC/TA3
  → Predicted: AQC  (confidence: 100%)

Saturday 10:00 → expect MP&S[H]
  → Predicted: AQC  (confidence: 33%)



In [4]:
#Save and download model:
import pickle
from google.colab import files

with open("knn_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved!")
files.download("knn_model.pkl")
# Move downloaded knn_model.pkl into same folder as app.py

Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>